In [1]:
import torch
import transformers

print("Transformers 版本:", transformers.__version__)
print("PyTorch 版本:", torch.__version__)
print("GPU 是否可用:", torch.cuda.is_available())

c:\Users\16579\.conda\envs\Test3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers 版本: 5.5.4
PyTorch 版本: 2.11.0+cpu
GPU 是否可用: False


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2-1.5B-Instruct"

print("正在加载模型...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype="auto"
)
print("模型加载完毕！可以开始实验了。")

正在加载模型...


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 3276.07it/s]


模型加载完毕！可以开始实验了。


In [3]:
def run_demo(question, system_p="你是一个助手。"):
    # 模拟生成
    inputs = tokenizer.apply_chat_template([{"role":"system","content":system_p},{"role":"user","content":question}], tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([inputs], return_tensors="pt").to(model.device)
    
    outputs = model.generate(**model_inputs, max_new_tokens=150, output_scores=True, return_dict_in_generate=True, do_sample=True, temperature=0.8)
    
    # 获取回答
    response = tokenizer.decode(outputs.sequences[0][len(model_inputs.input_ids[0]):], skip_special_tokens=True)
    
    # 模拟 SK-TUNING 指标
    transition_scores = model.compute_transition_scores(outputs.sequences, outputs.scores, normalize_logits=True)
    confidence = torch.exp(transition_scores[0]).mean().item()
    
    return response, confidence


# 

In [12]:
test_q = "请介绍一下 1996 年亚特兰大奥运会男子足球金牌得主——中国队的夺冠历程。"

print(f" 提问: {test_q}\n" + "="*50)

# 1. 原始回答
res, conf = run_demo(test_q)
print(f"[原始生成]: {res}")
print(f"[SK-TUNING 信心值]: {conf:.4f} (较低的值通常预示着幻觉)")

# 2. 自评估 (Self-Evaluation)
eval_q = f"刚才的回答是否有事实错误？请指出。回答内容：{res}"
eval_res, _ = run_demo(eval_q)
print(f"\n[Self-Evaluation 自评]: {eval_res}")

 提问: 请介绍一下 1996 年亚特兰大奥运会男子足球金牌得主——中国队的夺冠历程。
[原始生成]: 很遗憾，中国并没有在1996年亚特兰大奥运会上获得过男子足球金牌。中国男足在20世纪80年代末期和90年代初曾有过短暂的辉煌时期，并且在2004年的雅典奥运会上获得了男子足球的铜牌。但是，在1996年亚特兰大奥运会上，中国并未取得任何奖牌。

关于中国足球的历史和成就，我可以提供一些基本的事实：

- 1972年，中国首次参加了国际比赛——亚洲杯。
- 1984年中国男足第一次参加世界大赛——美国世界杯预选赛。
- 1985年中国男足第一次进入亚洲区世预
[SK-TUNING 信心值]: 0.6795 (较低的值通常预示着幻觉)

[Self-Evaluation 自评]: 没有事实错误。
